# Ultron OCR contest: 5 models, same 100 lines

Every model reads **the same 100 CATMuS test lines** (1 line in 50 of the test split, then 1 in 3: 5 languages, 10th–15th c.),
on the same GPU, scored with the same CER and WER.

```
same 100 lines ──▶ TrOCR base ─────┐
               ──▶ TrOCR cursiva ──┤
               ──▶ Qwen2-VL-2B ────┼──▶ saved line by line on Drive ──▶ leaderboard + chart ──▶ MLflow
               ──▶ Qwen3-VL-4B ────┤
               ──▶ MEDUSA 4B ──────┘
```

**How to run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.
Total about 45 minutes. Each answer is saved on Google Drive immediately: if Colab disconnects, **Run all again** and it continues where it stopped (finished models are skipped).

## 1. Install

In [ ]:
%pip install -q -U transformers accelerate peft datasets mlflow
%pip uninstall -q -y torchao

## 2. Save folder on Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
BASE_DIR = "/content/drive/MyDrive/ultron-mlflow"
PRED_DIR = f"{BASE_DIR}/contest_predictions"
os.makedirs(PRED_DIR, exist_ok=True)
print("Answers saved in", PRED_DIR)

## 3. The contestants

Remove a line from `CONTESTANTS` to skip a model.

In [ ]:
CONTESTANTS = {
    "trocr-base": dict(
        kind="trocr", model_id="medieval-data/trocr-medieval-base",
        note="In production today"),
    "trocr-cursiva": dict(
        kind="trocr", model_id="medieval-data/trocr-medieval-cursiva",
        note="TrOCR trained on one script family (cursiva)"),
    "qwen2-vl-2b-catmus": dict(
        kind="vlm", model_id="medieval-data/qwen2-vl-2b-catmus",
        prompt="Convert this image to text.",
        note="Small vision-language model fine-tuned on CATMuS (its training prompt)"),
    "qwen3-vl-4b-catmus": dict(
        kind="vlm", model_id="Qwen/Qwen3-VL-4B-Instruct", adapter="small-models-for-glam/Qwen3-VL-4B-catmus",
        prompt="Transcribe the text shown in this image.",
        note="Qwen3-VL 4B + LoRA adapter trained on CATMuS"),
    "medusa-0.2-4b": dict(
        kind="vlm", model_id="ENC-PSL/Medusa0.2Line-4B",
        prompt="Transcribe the handwritten text in this line image.\nOutput ONLY the transcription.",
        extra={"enable_thinking": False},
        note="Vision-language model trained for medieval line transcription"),
}
print(len(CONTESTANTS), "contestants:", ", ".join(CONTESTANTS))

## 4. The same 100 test lines

In [ ]:
import itertools
from datasets import load_dataset

ds = load_dataset("CATMuS/medieval", split="test", streaming=True)
rows300 = list(itertools.islice((r for i, r in enumerate(ds) if i % 50 == 0), 300))
sample = rows300[::3]
print(len(sample), "lines | languages:", sorted({r["language"] for r in sample}))

## 5. How each kind of model reads a line

In [ ]:
import gc, torch

TEMPLATE_FROM = {"medieval-data/qwen2-vl-2b-catmus": "Qwen/Qwen2-VL-2B-Instruct"}

# A vision-language model turns an image into many tokens: a very large line image can fill the
# 15 GB of a T4. Only images above this size are shrunk (aspect ratio kept); normal lines are untouched.
MAX_PIXELS = 300_000

def fit_image(image):
    w, h = image.size
    if w * h <= MAX_PIXELS:
        return image
    scale = (MAX_PIXELS / (w * h)) ** 0.5
    small = image.resize((max(1, int(w * scale)), max(1, int(h * scale))))
    print(f"  large image {w}x{h} shrunk to {small.size[0]}x{small.size[1]}")
    return small

def load_reader(cfg):
    """Returns (read_function, model) for one contestant."""
    if cfg["kind"] == "trocr":
        from transformers import TrOCRProcessor, VisionEncoderDecoderModel
        proc = TrOCRProcessor.from_pretrained(cfg["model_id"])
        model = VisionEncoderDecoderModel.from_pretrained(cfg["model_id"]).to("cuda").eval()
        def read(image):
            pv = proc(images=image, return_tensors="pt").pixel_values.to("cuda")
            with torch.inference_mode():
                ids = model.generate(pv, max_new_tokens=128, num_beams=4, no_repeat_ngram_size=4)
            return proc.batch_decode(ids, skip_special_tokens=True)[0].strip()
        return read, model

    from transformers import AutoProcessor, AutoModelForImageTextToText
    processor = AutoProcessor.from_pretrained(cfg["model_id"])
    if getattr(processor, "chat_template", None) is None:
        # Some fine-tuned models (e.g. qwen2-vl-2b-catmus) were saved without the chat template:
        # borrow it from the base model they were trained from.
        base = TEMPLATE_FROM.get(cfg["model_id"])
        if base is None:
            raise ValueError(f"{cfg['model_id']} has no chat template and no base model is known")
        processor.chat_template = AutoProcessor.from_pretrained(base).chat_template
        print(f"  chat template borrowed from {base}")
    model = AutoModelForImageTextToText.from_pretrained(cfg["model_id"], torch_dtype="auto", device_map="auto")
    if cfg.get("adapter"):
        import peft.tuners.lora.torchao as _lt
        _lt.is_torchao_available = lambda: False      # avoids the torchao/peft clash we hit before
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, cfg["adapter"])
    model.eval()
    def read(image):
        image = fit_image(image)
        msgs = [{"role": "user", "content": [{"type": "image", "image": image},
                                             {"type": "text", "text": cfg["prompt"]}]}]
        inputs = processor.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=True,
                                               return_tensors="pt", **cfg.get("extra", {})).to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
        text = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        del inputs, out
        torch.cuda.empty_cache()
        return " ".join(text.split())
    return read, model

print("ready")

## 6. Run the contest (continues where it stopped)

In [ ]:
import json, time

def pred_file(name): return f"{PRED_DIR}/{name}.jsonl"
def load_done(name):
    done = {}
    if os.path.exists(pred_file(name)):
        with open(pred_file(name), encoding="utf-8") as f:
            for line in f:
                r = json.loads(line); done[r["i"]] = r
    return done

gpu_peak = {}
for name, cfg in CONTESTANTS.items():
    done = load_done(name)
    if len(done) == len(sample):
        print(f"{name}: already finished, skipped"); continue
    print(f"\n=== {name}: {len(done)}/{len(sample)} done, loading model ...")
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    try:
        read, model = load_reader(cfg)
    except Exception as e:
        print(f"{name}: could not load ({type(e).__name__}: {e}). Skipped."); continue
    try:
        with open(pred_file(name), "a", encoding="utf-8") as f:
            for i, r in enumerate(sample):
                if i in done: continue
                t0 = time.time()
                pred = read(r["im"].convert("RGB"))
                torch.cuda.synchronize()
                rec = {"i": i, "language": r["language"], "truth": r["text"].strip(), "prediction": pred,
                       "seconds": time.time() - t0}
                f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
                done[i] = rec
                if len(done) % 20 == 0: print(f"  {len(done)}/{len(sample)}")
    except Exception as e:
        print(f"{name}: stopped at line {len(done)} ({type(e).__name__}: {e}). Moving to the next model.")
        del model, read; gc.collect(); torch.cuda.empty_cache(); continue
    gpu_peak[name] = torch.cuda.max_memory_allocated() / 1e9
    with open(f"{PRED_DIR}/{name}.gpu", "w") as f: f.write(str(gpu_peak[name]))
    print(f"{name}: finished, peak GPU {gpu_peak[name]:.1f} GB")
    del model, read; gc.collect(); torch.cuda.empty_cache()
print("\nContest finished")

## 7. Leaderboard

In [ ]:
import unicodedata
import pandas as pd

def norm(s): return " ".join(unicodedata.normalize("NFC", str(s)).split())
def ed(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]
def rates(ts, ps):
    ce = cn = we = wn = 0
    for t, p in zip(ts, ps):
        t, p = norm(t), norm(p)
        ce += ed(t, p); cn += len(t); we += ed(t.split(), p.split()); wn += len(t.split())
    return ce / cn, we / wn

results, tables = [], {}
for name, cfg in CONTESTANTS.items():
    done = load_done(name)
    if len(done) < len(sample):
        print(f"{name}: only {len(done)}/{len(sample)} lines, not ranked (run section 6 again)"); continue
    t = pd.DataFrame([done[i] for i in sorted(done)]); tables[name] = t
    cer, wer = rates(t.truth, t.prediction)
    gpu_path = f"{PRED_DIR}/{name}.gpu"
    gpu = float(open(gpu_path).read()) if os.path.exists(gpu_path) else float("nan")
    results.append({"model": name, "CER": cer, "WER": wer, "s_per_line": t.seconds.mean(), "GPU_GB": gpu,
                    **{f"CER {lang}": rates(g.truth, g.prediction)[0] for lang, g in t.groupby("language")}})

board = pd.DataFrame(results).sort_values("CER").reset_index(drop=True)
board.index += 1
pct = {c: "{:.1%}" for c in board.columns if c.startswith(("CER", "WER"))}
display(board.style.format({**pct, "s_per_line": "{:.2f}", "GPU_GB": "{:.1f}"}, na_rep="–"))

## 8. Chart: accuracy vs speed

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
ax = axes[0]
order = board.sort_values("CER", ascending=False)
bars = ax.barh(order.model, order.CER * 100,
               color=["#2f6db5" if m == "trocr-base" else "#9aa7b8" for m in order.model])
for b, v in zip(bars, order.CER):
    ax.text(b.get_width() + 0.3, b.get_y() + b.get_height() / 2, f"{v:.1%}", va="center")
ax.set_xlabel("Character error rate (%) — lower is better")
ax.set_title("Same 100 CATMuS lines (blue = model in production)")

ax = axes[1]
for row in board.itertuples():
    size = (row.GPU_GB if row.GPU_GB == row.GPU_GB else 2) * 60
    ax.scatter(row.s_per_line, row.CER * 100, s=size, alpha=0.7,
               color="#2f6db5" if row.model == "trocr-base" else "#9aa7b8")
    ax.annotate(f"{row.model}\n{row.CER:.1%}", (row.s_per_line, row.CER * 100),
                textcoords="offset points", xytext=(8, 8), fontsize=9)
ax.set_xlabel("Seconds per line (Colab T4)")
ax.set_ylabel("CER (%)")
ax.set_ylim(bottom=0)
ax.set_title("Accuracy vs speed (bubble = GPU memory)")
plt.tight_layout()
plt.savefig(f"{BASE_DIR}/contest_100_lines.png", dpi=150)
plt.show()

## 9. Examples: what each model wrote for the same lines

In [ ]:
pd.set_option("display.max_colwidth", 90)
for i in [0, 25, 50, 75]:
    print(f"--- line {i} ({sample[i]['language']}) ---")
    print(f"{'truth':>20}: {sample[i]['text'].strip()}")
    for name, t in tables.items():
        print(f"{name:>20}: {t.loc[t.i == i, 'prediction'].iloc[0]}")

## 10. Log every contestant in MLflow

Same database as the main MLflow notebook (`ultron_ocr_mlflow.ipynb` with `USE_DRIVE = True`), so the contest appears in its leaderboard and history.

In [ ]:
import os
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import mlflow
from datetime import datetime

mlflow.set_tracking_uri(f"sqlite:///{BASE_DIR}/mlflow.db")
if mlflow.get_experiment_by_name("ultron-ocr") is None:
    mlflow.create_experiment("ultron-ocr", artifact_location=f"{BASE_DIR}/artifacts")
exp = mlflow.set_experiment("ultron-ocr")
contest_id = "contest-100-" + datetime.now().strftime("%Y-%m-%d")
logged = {r.data.tags.get("contest_model") for r in mlflow.MlflowClient().search_runs(
    [exp.experiment_id], f"tags.contest_id = '{contest_id}'")}

for row in board.itertuples():
    if row.model in logged: continue
    cfg = CONTESTANTS[row.model]
    with mlflow.start_run(run_name=f"{row.model} {contest_id}"):
        mlflow.log_params({"model_id": cfg["model_id"] + (f" + {cfg['adapter']}" if cfg.get("adapter") else ""),
                           "model": row.model, "benchmark": "catmus-test-every50-third", "lines": len(sample),
                           "decoding": "beam search (num_beams=4, no_repeat_ngram_size=4)" if cfg["kind"] == "trocr" else "greedy",
                           "component": "recognizer", "segmenter": "none (line images)",
                           "prompt": cfg.get("prompt", "none (TrOCR)")})
        mlflow.set_tags({"source": "contest", "contest_id": contest_id, "contest_model": row.model,
                         "date": datetime.now().strftime("%Y-%m-%d"), "notes": cfg["note"]})
        mlflow.log_metrics({"cer": row.CER, "wer": row.WER, "s_per_line": row.s_per_line,
                            **({"gpu_gb": row.GPU_GB} if row.GPU_GB == row.GPU_GB else {})})
        mlflow.log_table(tables[row.model], "predictions.json")
print(f"Logged {len(board) - len(logged)} runs under contest '{contest_id}'")
print("Winner on CER:", board.iloc[0].model, f"({board.iloc[0].CER:.1%})")